In [28]:
!pip install wbgapi

In [29]:
import pandas as pd
import numpy as np
import os
import wbgapi as wb
from pathlib import Path
from google.colab import drive

# ============================================================================
# 1. CONFIGURATION (Path Settings)
# ============================================================================
# Mount Google Drive
drive.mount('/content/drive')

# Define folder paths
# Note: Adjust these paths to your specific directory structure
RAW_DATA_FOLDER = Path("/content/drive/MyDrive/Resilient_Housing_Global_Regression/data_raw/WB")
PROCESSED_DATA_FOLDER = Path("/content/drive/MyDrive/Resilient_Housing_Global_Regression/data_processed")
PROCESSED_DATA_FOLDER.mkdir(parents=True, exist_ok=True)

# Local file names
FILE_WGI = RAW_DATA_FOLDER / "wgidataset.xlsx"
FILE_DB_CONST = RAW_DATA_FOLDER / "Dealing with Construction Permits.xlsx"
FILE_CLASS = RAW_DATA_FOLDER / "CLASS.xlsx"
FILE_WB_API_BACKUP = PROCESSED_DATA_FOLDER / "raw_wb_api_backup.csv"

# Definition of WB indicators to fetch via API
# Mapping: World Bank Series Code -> Friendly Name
WB_API_INDICATORS = {
    # Economy & Population
    'NY.GDP.MKTP.PP.KD': 'GDP_PPP_Intl_2021',
    'NY.GDP.PCAP.PP.KD': 'GDP_PCAP_PPP_Intl_2021',
    'SP.POP.TOTL': 'Population_Total',
    'SP.URB.TOTL.IN.ZS': 'Urban_Population_Pct',

    # Poverty & Inequality
    'SI.SPR.PCAP': 'Poverty_Survey_Mean_Income_PPP',
    'SI.POV.UMIC': 'Poverty_HC_Ratio_at_USD8_30',
    'SI.POV.NAHC': 'Poverty_HC_Ratio_National_Line',
    'SI.POV.LMIC': 'Poverty_HC_Ratio_at_USD4_20',
    'SI.POV.DDAY' : 'Poverty_HC_Ratio_at_USD3_00',
    'SI.POV.GINI': 'Gini_Index',
    'EN.POP.SLUM.UR.ZS': 'Slum_Population_Urban_Pct',

    # Poverty Gaps
    'SI.POV.GAPS': 'Poverty_Gap_at_USD3_00_2021PPP',
    'SI.POV.LMIC.GP': 'Poverty_Gap_at_USD4_20_2021PPP',
    'SI.POV.UMIC.GP': 'Poverty_Gap_at_USD8_30_2021PPP',

    # Environment & Land
    'AG.LND.TOTL.K2': 'Land_Area_sq_km',
    'AG.LND.PRCP.MM': 'Average_Precipitation_mm_per_year',

    # Social Protection
    'IQ.SPI.ASSIST': 'Social_Assistance_Coverage_Pct'
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
def fetch_and_save_wb_data_target_year(output_path, indicators, target_year=2023):
    """
    Fetches the most recent value up to the target_year (e.g., 2023) from WB API.
    """
    print(f"--- Fetching WB API Data (Latest up to {target_year}) ---")

    # Define the time range from 2010 to target_year(2023)
    time_range = range(2010, target_year + 1)

    # Fetch data for all years in range
    df = wb.data.DataFrame(indicators.keys(), time=time_range, labels=True).reset_index()

    results = []
    for series, simple_name in indicators.items():
        df_series = df[df['series'] == series].copy()

        # Identify year columns (e.g., 'YR2015', 'YR2023')
        year_cols = [c for c in df_series.columns if c.startswith('YR')]

        if not year_cols:
            continue

        # Melt to long format to find the latest available year per country
        df_long = df_series.melt(id_vars=['economy', 'Country'], value_vars=year_cols,
                                 var_name='Year', value_name='Value')

        # Clean Year string and Drop NaNs
        df_long['Year'] = df_long['Year'].str.replace('YR', '').astype(int)
        df_long = df_long.dropna(subset=['Value'])

        # Sort by economy and Year (Descending) to get the most recent year <= target_year
        df_latest = df_long.sort_values(['economy', 'Year'], ascending=[True, False])
        df_latest = df_latest.groupby('economy').first().reset_index()

        # Rename columns to final format
        df_latest = df_latest.rename(columns={
            'Value': f'WB_{simple_name}_Value',
            'Year': f'WB_{simple_name}_Year',
            'economy': 'Country Code',
            'Country': 'Country Name'
        })
        results.append(df_latest[['Country Code', 'Country Name', f'WB_{simple_name}_Value', f'WB_{simple_name}_Year']])

    # Merge all indicators together
    wb_final = results[0]
    for r in results[1:]:
        wb_final = pd.merge(wb_final, r, on=['Country Code', 'Country Name'], how='outer')

    # Save to CSV
    wb_final.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ Success: API data (up to {target_year}) saved to {output_path}")
    return wb_final

# 🚀 Check if file exists, if not, fetch from API
if FILE_WB_API_BACKUP.exists():
    print(f"📊 Loading existing API data from: {FILE_WB_API_BACKUP}")
    df_wb_api_raw = pd.read_csv(FILE_WB_API_BACKUP)
else:
    print("🌐 API data not found. Fetching from World Bank API...")
    df_wb_api_raw = fetch_and_save_wb_data_target_year(FILE_WB_API_BACKUP, WB_API_INDICATORS, target_year=2023)

📊 Loading existing API data from: /content/drive/MyDrive/Resilient_Housing_Global_Regression/data_processed/raw_wb_api_backup.csv


In [31]:
# Target Country List (Used to filter out aggregates/regions)
country_list = [
    'Aruba', 'Afghanistan', 'Angola', 'Albania', 'Andorra', 'United Arab Emirates', 'Argentina',
    'Armenia', 'American Samoa', 'Antigua and Barbuda', 'Australia', 'Austria', 'Azerbaijan',
    'Burundi', 'Belgium', 'Benin', 'Burkina Faso', 'Bangladesh', 'Bulgaria', 'Bahrain',
    'Bahamas, The', 'Bosnia and Herzegovina', 'Belarus', 'Belize', 'Bermuda', 'Bolivia',
    'Brazil', 'Barbados', 'Brunei Darussalam', 'Bhutan', 'Botswana', 'Central African Republic',
    'Canada', 'Switzerland', 'Channel Islands', 'Chile', 'Chad','China', "Cote d'Ivoire", 'Cameroon',
    'Congo, Dem. Rep.', 'Congo, Rep.', 'Colombia', 'Comoros', 'Cabo Verde', 'Costa Rica',
    'Cuba', 'Curacao', 'Cayman Islands', 'Cyprus', 'Czechia', 'Germany', 'Djibouti', 'Dominica',
    'Denmark', 'Dominican Republic', 'Algeria', 'Ecuador', 'Egypt, Arab Rep.', 'Eritrea', 'Spain',
    'Estonia', 'Ethiopia', 'Finland', 'Fiji', 'France', 'Faroe Islands', 'Micronesia, Fed. Sts.',
    'Gabon', 'United Kingdom', 'Georgia', 'Ghana', 'Gibraltar', 'Guinea', 'Gambia, The',
    'Guinea-Bissau', 'Equatorial Guinea', 'Greece', 'Grenada', 'Greenland', 'Guatemala',
    'Guam', 'Guyana', 'Hong Kong SAR, China', 'Honduras', 'Croatia', 'Haiti', 'Hungary',
    'Indonesia', 'Isle of Man', 'India', 'Ireland', 'Iran, Islamic Rep.', 'Iraq', 'Iceland',
    'Israel', 'Italy', 'Jamaica', 'Jordan', 'Japan', 'Kazakhstan', 'Kenya', 'Kyrgyz Republic',
    'Cambodia', 'Kiribati', 'St. Kitts and Nevis', 'Korea, Rep.', 'Kuwait', 'Lao PDR', 'Lebanon',
    'Liberia', 'Libya', 'St. Lucia', 'Liechtenstein', 'Sri Lanka', 'Lesotho', 'Lithuania',
    'Luxembourg', 'Latvia', 'Macao SAR, China', 'St. Martin (French part)', 'Morocco', 'Monaco',
    'Moldova', 'Madagascar', 'Maldives', 'Mexico', 'Marshall Islands', 'North Macedonia', 'Mali',
    'Malta', 'Myanmar', 'Montenegro', 'Mongolia', 'Northern Mariana Islands', 'Mozambique',
    'Mauritania', 'Mauritius', 'Malawi', 'Malaysia', 'Namibia', 'New Caledonia', 'Niger', 'Nigeria',
    'Nicaragua', 'Netherlands', 'Norway', 'Nepal', 'Nauru', 'New Zealand', 'Oman', 'Pakistan',
    'Panama', 'Peru', 'Philippines', 'Palau', 'Papua New Guinea', 'Poland', 'Puerto Rico (US)',
    'Korea, Dem. People\'s Rep.', 'Portugal', 'Paraguay', 'West Bank and Gaza', 'French Polynesia',
    'Qatar', 'Romania', 'Russian Federation', 'Rwanda', 'Saudi Arabia', 'Sudan', 'Senegal',
    'Singapore', 'Solomon Islands', 'Sierra Leone', 'El Salvador', 'San Marino', 'Somalia, Fed. Rep.',
    'Serbia', 'South Sudan', 'Sao Tome and Principe', 'Suriname', 'Slovak Republic', 'Slovenia',
    'Sweden', 'Eswatini', 'Sint Maarten (Dutch part)', 'Seychelles', 'Syrian Arab Republic',
    'Turks and Caicos Islands', 'Togo', 'Thailand', 'Tajikistan', 'Turkmenistan', 'Timor-Leste',
    'Tonga', 'Trinidad and Tobago', 'Tunisia', 'Turkiye', 'Tuvalu', 'Tanzania', 'Uganda', 'Ukraine',
    'Uruguay', 'United States', 'Uzbekistan', 'St. Vincent and the Grenadines', 'Venezuela, RB',
    'British Virgin Islands', 'Virgin Islands (U.S.)', 'Viet Nam', 'Vanuatu', 'Samoa', 'Kosovo',
    'Yemen, Rep.', 'South Africa', 'Zambia', 'Zimbabwe',
]

In [32]:
# --- Helper Functions Definition ---

def load_and_clean_region_data(file_path, country_list):
    df = pd.read_excel(file_path, header=0)
    df_region = df[['Economy', 'Region']].copy()
    df_region.columns = ['Country Name', 'Region']
    df_final = df_region[df_region['Country Name'].isin(country_list)].copy()
    df_final.drop_duplicates(subset=['Country Name'], inplace=True)
    return df_final

def load_and_clean_wgi(file_path, country_list):
    df = pd.read_excel(file_path, header=0)
    df = df[['countryname', 'year', 'indicator', 'estimate']].copy()
    df.columns = ['Country Name', 'Year', 'Indicator', 'Value']
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    df_filtered = df[df['Country Name'].isin(country_list)].copy()
    df_filtered.dropna(subset=['Value'], inplace=True)
    df_filtered.sort_values(by=['Country Name', 'Indicator', 'Year'], ascending=[True, True, False], inplace=True)
    df_latest = df_filtered.groupby(['Country Name', 'Indicator']).first().reset_index()

    df_latest['Indicator_Latest_Value'] = 'WGI_' + df_latest['Indicator'] + '_Value'
    df_latest['Indicator_Latest_Year'] = 'WGI_' + df_latest['Indicator'] + '_Year'
    df_val = df_latest.pivot_table(index='Country Name', columns='Indicator_Latest_Value', values='Value').reset_index()
    df_yr = df_latest.pivot_table(index='Country Name', columns='Indicator_Latest_Year', values='Year').reset_index()
    return pd.merge(df_val, df_yr, on='Country Name', how='outer')

def load_and_clean_db_const(file_path, country_list):
    df = pd.read_excel(file_path, header=None)
    df_data = df.iloc[10:].copy()
    df_subset = df_data[[1, 2, 6]].copy()
    df_subset.columns = ['Country Name', 'DB_CONST_SCORE', 'DB_CONST_QUALITY_INDEX']
    df_subset['DB_CONST_SCORE'] = pd.to_numeric(df_subset['DB_CONST_SCORE'], errors='coerce')
    df_subset['DB_CONST_QUALITY_INDEX'] = pd.to_numeric(df_subset['DB_CONST_QUALITY_INDEX'], errors='coerce')
    df_final = df_subset[df_subset['Country Name'].isin(country_list)].copy()

    df_final.loc[(df_final['DB_CONST_QUALITY_INDEX'] > 15) | (df_final['DB_CONST_QUALITY_INDEX'] < 0), 'DB_CONST_QUALITY_INDEX'] = np.nan
    df_final.loc[df_final['DB_CONST_SCORE'] == 0, 'DB_CONST_SCORE'] = np.nan
    return df_final

In [35]:
def process_final_integration(df_api, country_list):
    """
    Integrates WB API data with local datasets (WGI, DB, Region) and formats column names.
    Note: Requires load_and_clean_wgi, load_and_clean_db_const, and load_and_clean_region_data functions.
    """
    print("--- Processing Final Integration ---")

    # 1. Filter by country list (Remove regions and aggregates)
    df_filtered = df_api[df_api['Country Name'].isin(country_list)].copy()

    # 2. Load Local Data (Passing country_list to resolve the previous TypeError)
    # WGI Data
    df_wgi = load_and_clean_wgi(RAW_DATA_FOLDER / "wgidataset.xlsx", country_list)

    # Doing Business Data
    df_db_const = load_and_clean_db_const(RAW_DATA_FOLDER / "Dealing with Construction Permits.xlsx", country_list)

    # Regional Classification Data
    df_region = load_and_clean_region_data(RAW_DATA_FOLDER / "CLASS.xlsx", country_list)

    # 3. Merge Datasets
    # Merge Region first, then WGI and DB using 'Country Name' as the key
    df_final = pd.merge(df_filtered, df_region[['Country Name', 'Region']], on='Country Name', how='left')
    df_final = pd.merge(df_final, df_wgi, on='Country Name', how='left')
    df_final = pd.merge(df_final, df_db_const, on='Country Name', how='left')

    # 4. Rename Columns using the provided maps
    # Define renaming dictionaries
    WGI_RENAME_MAP = {
        'WGI_va_Value': 'WB_WGI_Voice_Accountability_Value',
        'WGI_pv_Value': 'WB_WGI_Political_Stability_Value',
        'WGI_ge_Value': 'WB_WGI_Government_Effectiveness_Value',
        'WGI_rq_Value': 'WB_WGI_Regulatory_Quality_Value',
        'WGI_rl_Value': 'WB_WGI_Rule_of_Law_Value',
        'WGI_cc_Value': 'WB_WGI_Control_of_Corruption_Value',
        'WGI_va_Year': 'WB_WGI_Voice_Accountability_Year',
        'WGI_pv_Year': 'WB_WGI_Political_Stability_Year',
        'WGI_ge_Year': 'WB_WGI_Government_Effectiveness_Year',
        'WGI_rq_Year': 'WB_WGI_Regulatory_Quality_Year',
        'WGI_rl_Year': 'WB_WGI_Rule_of_Law_Year',
        'WGI_cc_Year': 'WB_WGI_Control_of_Corruption_Year',
    }

    DB_RENAME_MAP = {
        'DB_CONST_SCORE': 'WB_DB_Construction_Permits_Score',
        'DB_CONST_QUALITY_INDEX': 'WB_DB_Building_Quality_Index',
    }

    # Combine maps and rename
    df_final = df_final.rename(columns={**WGI_RENAME_MAP, **DB_RENAME_MAP})

    # 5. Final Formatting & Column Ordering
    # Set CountryName as the index
    df_final = df_final.rename(columns={'Country Name': 'CountryName'}).set_index('CountryName')

    # Define the output column order (Base -> WB API -> WGI -> DB)
    base_cols = ['Country Code', 'Region']

    # WB API Indicators (excluding WGI and DB related columns)
    wb_api_cols = sorted([c for c in df_final.columns if c.startswith('WB_') and 'WGI' not in c and 'DB' not in c])

    # WGI and DB Indicators
    wgi_cols = sorted([c for c in df_final.columns if 'WGI' in c])
    db_cols = sorted([c for c in df_final.columns if 'DB' in c])

    final_order = base_cols + wb_api_cols + wgi_cols + db_cols
    df_output = df_final[[c for c in final_order if c in df_final.columns]]

    # 6. Export to CSV
    output_csv = PROCESSED_DATA_FOLDER / "Country_WorldBank_Data_Cleaned_For_Merge.csv"
    df_output.to_csv(output_csv, encoding='utf-8-sig')

    print(f"🎉 Final dataset ready and exported to: {output_csv}")
    print(f"Total columns: {len(df_output.columns)}")

    return df_output


In [36]:
# Execution
df_final_result = process_final_integration(df_wb_api_raw, COUNTRY_LIST)

--- Processing Final Integration ---
🎉 Final dataset ready and exported to: /content/drive/MyDrive/Resilient_Housing_Global_Regression/data_processed/Country_WorldBank_Data_Cleaned_For_Merge.csv
Total columns: 50
